# Module 04 Exercise: Coding with AI

Fix the `monthly_mean` function below so it stays lazy (Dask-backed)
instead of forcing the whole array into memory, verify the fix with
a real test, then explain the bug in your own words.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))
from grading import grade

import numpy as np
import xarray as xr
from dask.base import is_dask_collection

## What a passing answer needs

The locked cell below is what the automated grader checks your answer
against. Read it before you start, and run it.

In [ ]:
question = (
    "The learner was asked to explain, in their own words, why calling "
    "`.values` inside monthly_mean (before returning) was a problem for "
    "a dask-backed xarray pipeline."
)
rubric = (
    "Pass only if the explanation mentions that `.values` forces eager "
    "computation / converts the lazy Dask-backed array to a NumPy array "
    "immediately, which can load the entire (possibly huge) dataset into "
    "memory at once, rather than keeping it chunked/lazy until `.compute()` "
    "is explicitly called."
)

## The buggy function

Here's the function from the README. Fix the marked line so the
return value stays a lazy Dask array (hint: you'll also need to
change the `.mean(...)` call, since a lazy `xarray.DataArray` uses
`dim=`, not `axis=`).

In [ ]:
def monthly_mean(path):
    ds = xr.open_mfdataset(
        path,
        chunks={"time": 30, "lat": 100, "lon": 100},
    )
    temperature = ds["temperature"].values  # TODO: fix this line
    return temperature.mean(axis=0)

## Test your fix

Run the cell below. It builds two small sample netCDF files, calls
your `monthly_mean`, and checks that the result is still a lazy
Dask array before `.compute()` is called.

In [ ]:
import tempfile


def test_monthly_mean_remains_lazy(tmp_dir):
    for i in range(2):
        ds = xr.Dataset(
            {"temperature": (("time", "lat", "lon"), np.full((2, 3, 4), i + 1.0))},
            coords={
                "time": np.arange(i * 2, i * 2 + 2),
                "lat": np.arange(3),
                "lon": np.arange(4),
            },
        )
        ds.to_netcdf(tmp_dir / f"temperature_{i}.nc")

    result = monthly_mean(str(tmp_dir / "temperature_*.nc"))
    assert is_dask_collection(result.data), "Expected a lazy Dask array before .compute()"

    computed = result.compute()
    assert isinstance(computed.data, np.ndarray)
    np.testing.assert_allclose(computed.values, 1.5)

    print("PASS: monthly_mean stayed lazy until .compute()")


with tempfile.TemporaryDirectory() as tmp:
    test_monthly_mean_remains_lazy(Path(tmp))

## Explain the bug

In 1-2 sentences, explain why the original `.values` line was a
problem.

In [ ]:
explanation = "TODO: your explanation here"

## Check your explanation

In [ ]:
result = grade(question, rubric, explanation)
print("PASS" if result["pass"] else "NEEDS WORK")
print(result["feedback"])